In [125]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as ply

In [2]:
%pip install pymysql

Note: you may need to restart the kernel to use updated packages.


In [126]:
import getpass
password = getpass.getpass("Enter your MySQL root password: ")

In [165]:
import pymysql

connection = pymysql.connect(
    host='localhost',        
    user='root',
    password=password,
    database='credit_risk_project'
)


query = "SELECT * FROM customer_features WHERE customer_id BETWEEN 5001 AND 10000;"

df = pd.read_sql(query, connection)
          
connection.close()

print(df.shape)
print(df.head())


C:\Users\ranve\AppData\Local\Temp\ipykernel_41076\3222484692.py:13: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, connection)


(5000, 12)
   customer_id  age    income employment_type region  num_loans  \
0         5001   62   90010.0        Salaried   West          2   
1         5002   62   42819.0        Salaried   East          2   
2         5003   48   32977.0      Unemployed   West          2   
3         5004   46  134095.0   Self-Employed  South          2   
4         5005   63   42843.0        Salaried   East          2   

   loan_amount_total  avg_interest_rate  avg_tenure_months  num_repayments  \
0          970651.48             15.850               27.0              16   
1          349082.20             10.660               24.0              16   
2          351057.52             12.060               42.0              16   
3         1372736.13              8.145               36.0              16   
4          675346.31             14.530               42.0              16   

   total_paid_amount  late_payment_ratio  
0          307602.71               0.125  
1          167261.46           

In [166]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix,accuracy_score
from xgboost import XGBClassifier
from imblearn.over_sampling import SMOTE

In [167]:
df['risk_label'] = np.where(
    df['late_payment_ratio'].isna(),
    np.nan,
    (df['late_payment_ratio'] > 0.4).astype(int)
)

print("Risk label distribution:")
print(df['risk_label'].value_counts(dropna=False))

print("\nMissing risk labels:", df['risk_label'].isna().sum())

Risk label distribution:
risk_label
1.0    2947
0.0    2053
Name: count, dtype: int64

Missing risk labels: 0


In [168]:
df_full = df.copy()

df = df.dropna(subset=['risk_label']).copy()

df = df.drop(columns=['customer_id', 'late_payment_ratio'])

print("Rows with known risk labels:", len(df))
print("Remaining missing risk labels:", df['risk_label'].isna().sum())
print("\nTarget distribution:")
print(df['risk_label'].value_counts())

Rows with known risk labels: 5000
Remaining missing risk labels: 0

Target distribution:
risk_label
1.0    2947
0.0    2053
Name: count, dtype: int64


In [169]:
print("Original records:", len(df_full))
print("Labeled records:", len(df))
print("Missing labels remaining:", df["risk_label"].isna().sum())

Original records: 5000
Labeled records: 5000
Missing labels remaining: 0


In [170]:
from sklearn.model_selection import train_test_split

X = df.drop(columns=["risk_label"]).copy()
y = df["risk_label"].astype(int).copy()

print("Features:", X.shape)
print("\nOriginal class distribution:")
print(y.value_counts())
print("\nClass percentages:")
print((y.value_counts(normalize=True) * 100).round(2))

Features: (5000, 10)

Original class distribution:
risk_label
1    2947
0    2053
Name: count, dtype: int64

Class percentages:
risk_label
1    58.94
0    41.06
Name: proportion, dtype: float64


In [171]:
print(y.value_counts())
print(y.value_counts(normalize=True).round(3))

risk_label
1    2947
0    2053
Name: count, dtype: int64
risk_label
1    0.589
0    0.411
Name: proportion, dtype: float64


In [172]:
# X = X.fillna(X.median(numeric_only=True))
# X = X.fillna("Unknown")
# X = pd.get_dummies(X, drop_first=True)

In [173]:
# print("Feature matrix shape:", X.shape)
# print(X.head())
# print("Missing feature values:", X.isna().sum().sum())

In [174]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=42,
    stratify=y
)

print("Training shape:", X_train.shape)
print("Testing shape:", X_test.shape)

print("\nTraining labels:")
print(y_train.value_counts())

print("\nTesting labels:")
print(y_test.value_counts())

Training shape: (3500, 10)
Testing shape: (1500, 10)

Training labels:
risk_label
1    2063
0    1437
Name: count, dtype: int64

Testing labels:
risk_label
1    884
0    616
Name: count, dtype: int64


In [175]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

numeric_cols = X_train.select_dtypes(include=np.number).columns.tolist()
categorical_cols = X_train.select_dtypes(
    exclude=np.number
).columns.tolist()

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="constant",
                               fill_value="Unknown")),
    ("encoder", OneHotEncoder(
        handle_unknown="ignore",
        sparse_output=False
    ))
])

preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_cols),
    ("categorical", categorical_pipeline, categorical_cols)
])

X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

print("Processed training shape:", X_train_processed.shape)
print("Processed testing shape:", X_test_processed.shape)
print("Missing values in training:",
      np.isnan(X_train_processed).sum())
print("Missing values in testing:",
      np.isnan(X_test_processed).sum())

Processed training shape: (3500, 15)
Processed testing shape: (1500, 15)
Missing values in training: 0
Missing values in testing: 0


In [176]:
from imblearn.over_sampling import SMOTE

print("Before SMOTE:")
print(y_train.value_counts())

smote = SMOTE(random_state=42)
X_train_balanced, y_train_balanced = smote.fit_resample(
    X_train_processed,
    y_train
)

print("\nAfter SMOTE:")
print(y_train_balanced.value_counts())

print("\nBalanced training shape:", X_train_balanced.shape)

Before SMOTE:
risk_label
1    2063
0    1437
Name: count, dtype: int64

After SMOTE:
risk_label
0    2063
1    2063
Name: count, dtype: int64

Balanced training shape: (4126, 15)


In [177]:
print(X_train_balanced)

[[-0.49576758  0.61512623  0.         ...  0.          0.
   1.        ]
 [-1.66945043 -0.56747815  0.         ...  0.          0.
   0.        ]
 [-1.60041026 -0.32936785  0.         ...  0.          1.
   0.        ]
 ...
 [-1.06499894  0.15350265  0.         ...  0.          0.
   1.        ]
 [-1.24225205  0.21783714  0.         ...  0.          1.
   0.        ]
 [ 0.43467028 -0.28533149  0.         ...  1.          0.
   0.        ]]


In [178]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.metrics import accuracy_score

lr = LogisticRegression(max_iter=1000, random_state=42)

lr.fit(X_train_balanced, y_train_balanced)

lr_preds = lr.predict(X_test_processed)

print("Logistic Regression Accuracy:", accuracy_score(y_test, lr_preds))
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, lr_preds))

print("\nClassification Report:")
print(classification_report(y_test, lr_preds, zero_division=0))

Logistic Regression Accuracy: 0.8466666666666667

Confusion Matrix:
[[498 118]
 [112 772]]

Classification Report:
              precision    recall  f1-score   support

           0       0.82      0.81      0.81       616
           1       0.87      0.87      0.87       884

    accuracy                           0.85      1500
   macro avg       0.84      0.84      0.84      1500
weighted avg       0.85      0.85      0.85      1500



In [179]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score
)

rf = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

rf.fit(X_train_balanced, y_train_balanced)

rf_preds = rf.predict(X_test_processed)

print("Random Forest Accuracy:", accuracy_score(y_test, rf_preds))
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, rf_preds))
print("\nClassification Report:")
print(classification_report(y_test, rf_preds, zero_division=0))

Random Forest Accuracy: 0.872

Confusion Matrix:
[[501 115]
 [ 77 807]]

Classification Report:
              precision    recall  f1-score   support

           0       0.87      0.81      0.84       616
           1       0.88      0.91      0.89       884

    accuracy                           0.87      1500
   macro avg       0.87      0.86      0.87      1500
weighted avg       0.87      0.87      0.87      1500



In [180]:
from xgboost import XGBClassifier
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    classification_report
)

model = XGBClassifier(
    n_estimators=300,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.85,
    colsample_bytree=0.85,
    random_state=42,
    eval_metric="logloss"
)

model.fit(X_train_balanced, y_train_balanced)

xgb_preds = model.predict(X_test_processed)

print("XGBoost Accuracy:", accuracy_score(y_test, xgb_preds))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, xgb_preds))

print("\nClassification Report:")
print(classification_report(y_test, xgb_preds, zero_division=0))

XGBoost Accuracy: 0.872

Confusion Matrix:
[[498 118]
 [ 74 810]]

Classification Report:
              precision    recall  f1-score   support

           0       0.87      0.81      0.84       616
           1       0.87      0.92      0.89       884

    accuracy                           0.87      1500
   macro avg       0.87      0.86      0.87      1500
weighted avg       0.87      0.87      0.87      1500



In [181]:
best_model = model

print("Selected model:", type(best_model).__name__)
print("Training samples:", len(y_train_balanced))
print("Test samples:", len(y_test))

Selected model: XGBClassifier
Training samples: 4126
Test samples: 1500


In [182]:
# Prepare all customers' features
X_all = df_full.drop(
    columns=["customer_id", "risk_label"],
    errors="ignore"
).copy()

# Apply the preprocessing fitted on training data
X_all_processed = preprocessor.transform(X_all)

# Predict probability of high risk and predicted class
risk_probs = best_model.predict_proba(X_all_processed)[:, 1]
risk_labels = best_model.predict(X_all_processed)

# Create results table
results = pd.DataFrame({
    "customer_id": df_full["customer_id"].values,
    "credit_risk_score": risk_probs.round(4),
    "risk_label": risk_labels,
    "prediction_date": pd.Timestamp.today().date()
})

print(results.head(10))
print("\nTotal predictions:", len(results))
print("Unique customers:", results["customer_id"].nunique())

print("\nPredicted risk distribution:")
print(results["risk_label"].value_counts())

   customer_id  credit_risk_score  risk_label prediction_date
0         5001             0.0980           0      2026-10-17
1         5002             0.8855           1      2026-10-17
2         5003             0.7978           1      2026-10-17
3         5004             0.0960           0      2026-10-17
4         5005             0.7980           1      2026-10-17
5         5006             0.9599           1      2026-10-17
6         5007             0.7928           1      2026-10-17
7         5008             0.7787           1      2026-10-17
8         5009             0.1821           0      2026-10-17
9         5010             0.9204           1      2026-10-17

Total predictions: 5000
Unique customers: 5000

Predicted risk distribution:
risk_label
1    3058
0    1942
Name: count, dtype: int64


In [183]:
print("Missing scores:", results["credit_risk_score"].isna().sum())
print("Scores within [0, 1]:",
      results["credit_risk_score"].between(0, 1).all())

Missing scores: 0
Scores within [0, 1]: True


In [184]:



connection = pymysql.connect(
    host="localhost",
    user="root",
    password=password,
    database="credit_risk_project"
)

try:
    with connection.cursor() as cursor:
        # Preserve the existing table structure.
        # Replace its data without dropping the table.
        cursor.execute("DELETE FROM credit_scores")

        sql = """
            INSERT INTO credit_scores
            (customer_id, credit_risk_score, risk_label, prediction_date)
            VALUES (%s, %s, %s, %s)
        """

        rows = [
            (
                int(row.customer_id),
                float(row.credit_risk_score),
                int(row.risk_label),
                row.prediction_date
            )
            for row in results.itertuples(index=False)
        ]

        cursor.executemany(sql, rows)

    connection.commit()
    print(f"Successfully saved {len(rows)} predictions.")

except Exception:
    connection.rollback()
    raise

finally:
    connection.close()


Successfully saved 5000 predictions.
